Build only on Linux x86_64 / Python 3.12. Disabled by default. This is a new artifact: ZIP, manifest and tracksdata wheel SHA may differ from historical locks. The second Cell verifies the new ZIP against the manifest just built and prints VERIFIED_WHEELHOUSE_LOCK. Persist the ZIP and this full block outside the Session, then paste the block into notebook 02 before running. Source manifest and pinned versions remain unchanged. This does not prove offline installation or authorize upload.


In [ ]:
ALLOW_WHEELHOUSE_BUILD = True
assert ALLOW_WHEELHOUSE_BUILD, "Build disabled; recover cached artifacts first"

# ============================================================
# Notebook 14 / 14.15D-1
# Build a Kaggle Linux / Python 3.12 offline dependency overlay.
#
# Downloads pinned wheels and builds tracksdata from the exact
# source commit. Does NOT install, import project code, infer,
# track, evaluate, or read GT.
# ============================================================

import sys
import json
import hashlib
import subprocess
import shutil
import zipfile
from pathlib import Path
from importlib import metadata as importlib_metadata

assert sys.platform == "linux"
assert sys.version_info[:2] == (3, 12), (
    f"Expected Kaggle Python 3.12, got {sys.version}"
)

WORKING = Path("/kaggle/working")
assert WORKING.is_dir()

BUNDLE = Path(
    "/kaggle/input/datasets/markdd520/"
    "biohub-frozen-v9-warm2-bundle/bundle"
)
assert (BUNDLE / "bundle_manifest.json").is_file()

OUT = WORKING / "biohub_wheelhouse_py312_v1"
WHEELS = OUT / "wheels"

ZIP_PATH = WORKING / "biohub_wheelhouse_py312_v1.zip"

# Never overwrite an earlier build.
assert not OUT.exists(), f"Output already exists: {OUT}"
assert not ZIP_PATH.exists(), f"ZIP already exists: {ZIP_PATH}"

TRACKSDATA_COMMIT = (
    "7bfeaf845ceb951226f19b72fe5b80e01601018a"
)

TRACKSDATA_URL = (
    "git+https://github.com/royerlab/tracksdata@"
    + TRACKSDATA_COMMIT
)

# Packages actually added or aligned during the successful
# Kaggle import-repair sequence.
# Base packages such as torch/numpy/pandas are intentionally
# not downloaded here; they will be checked in a clean runtime.

PINNED = [
    "zarr==3.1.6",
    "numcodecs==0.15.1",
    "donfig==0.8.1.post1",
    "geff==1.3.0.1.2",
    "geff-spec==1.2.0",
    "polars==1.43.2",
    "polars-runtime-32==1.43.2",
    "bidict==0.23.1",
    "ilpy==0.6.0",
    "pyscipopt==6.2.1",
    "rustworkx==0.18.1",
    "imagecodecs==2026.3.6",
]

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

def run_checked(command, label):
    print(f"\n========== {label} ==========", flush=True)
    print("COMMAND =", command, flush=True)

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
    )

    print(result.stdout[-20000:], flush=True)
    print("EXIT_CODE =", result.returncode, flush=True)

    assert result.returncode == 0, (
        f"{label} failed. Stop; preserve the partial build."
    )

OUT.mkdir(parents=False, exist_ok=False)
WHEELS.mkdir(parents=False, exist_ok=False)

# ------------------------------------------------------------
# 1. Download known pinned Linux-compatible binary wheels
# ------------------------------------------------------------

run_checked(
    [
        sys.executable,
        "-m",
        "pip",
        "download",
        "--no-deps",
        "--only-binary=:all:",
        "--dest",
        str(WHEELS),
        *PINNED,
    ],
    "DOWNLOAD PINNED WHEELS",
)

# ------------------------------------------------------------
# 2. Build the exact source-pinned tracksdata wheel
# ------------------------------------------------------------

run_checked(
    [
        sys.executable,
        "-m",
        "pip",
        "wheel",
        "--no-deps",
        "--wheel-dir",
        str(WHEELS),
        TRACKSDATA_URL,
    ],
    "BUILD TRACKSDATA",
)

# ------------------------------------------------------------
# 3. Verify wheel inventory and tracksdata metadata
# ------------------------------------------------------------

wheel_files = sorted(WHEELS.glob("*.whl"))

assert len(wheel_files) == len(PINNED) + 1, (
    f"Expected {len(PINNED) + 1} wheels, "
    f"found {len(wheel_files)}"
)

tracksdata_files = [
    path
    for path in wheel_files
    if path.name.lower().startswith("tracksdata-")
]

assert len(tracksdata_files) == 1, (
    "Expected exactly one tracksdata wheel"
)

tracksdata_wheel = tracksdata_files[0]

with zipfile.ZipFile(tracksdata_wheel) as archive:
    metadata_names = [
        name for name in archive.namelist()
        if name.endswith(".dist-info/METADATA")
    ]

    assert len(metadata_names) == 1

    metadata_text = archive.read(
        metadata_names[0]
    ).decode("utf-8")

expected_version = "0.1.0rc9.dev4+g7bfeaf845"

assert f"Version: {expected_version}" in metadata_text, (
    "tracksdata wheel version does not match the frozen commit"
)

# Record the newly produced wheel identity.
# Do not substitute either historical SHA.
tracksdata_sha = sha256_file(tracksdata_wheel)

print("\n========== TRACKSDATA IDENTITY ==========")
print("GIT_URL =", TRACKSDATA_URL)
print("WHEEL =", tracksdata_wheel.name)
print("VERSION =", expected_version)
print("NEW_WHEEL_SHA256 =", tracksdata_sha)

print("HISTORICAL_KAGGLE_WHEEL_SHA256 =",
      "9e5ecf8cc4e297c55515e836109c34eaee7103a27b73563909db06e0b0bea8c4")

print("LOCAL_CODEX_WHEEL_SHA256 =",
      "a12b65a10ab50f1449c532de72c8bf0c1cf0495088915dc93d4a30db93071df8")

# ------------------------------------------------------------
# 4. Produce an auditable manifest
# ------------------------------------------------------------

wheel_records = []

for path in wheel_files:
    record = {
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    wheel_records.append(record)

    print(
        "WHEEL",
        record["filename"],
        "BYTES",
        record["size_bytes"],
        "SHA256",
        record["sha256"],
    )

manifest = {
    "stage": "14.15D-1",
    "purpose": "kaggle_py312_offline_dependency_overlay",
    "python": "3.12",
    "platform": "linux_x86_64",
    "source_bundle_manifest_sha256":
        "615c3550cba46a5032344549d2769201af22c0425beb9349726874669ae9c865",
    "tracksdata_git_url": TRACKSDATA_URL,
    "tracksdata_commit": TRACKSDATA_COMMIT,
    "tracksdata_wheel_sha256": tracksdata_sha,
    "pinned_requirements": PINNED,
    "wheels": wheel_records,
    "torch_numpy_pandas_expected_from_base": {
        "torch": "2.10.0+cu128",
        "numpy": "2.0.2",
        "pandas": "2.3.3",
    },
    "status": "BUILT_NOT_OFFLINE_TESTED",
}

manifest_path = OUT / "wheelhouse_manifest.json"

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    ) + "\n",
    encoding="utf-8",
)

# ------------------------------------------------------------
# 5. Create one portable ZIP for private Kaggle Dataset upload
# ------------------------------------------------------------

created = shutil.make_archive(
    str(ZIP_PATH.with_suffix("")),
    "zip",
    root_dir=OUT,
)

assert Path(created) == ZIP_PATH
assert ZIP_PATH.is_file()

print("\n========== 14.15D-1 RESULT ==========")
print("WHEEL_COUNT =", len(wheel_files))
print("MANIFEST =", manifest_path)
print("MANIFEST_SHA256 =", sha256_file(manifest_path))
print("ZIP =", ZIP_PATH)
print("ZIP_BYTES =", ZIP_PATH.stat().st_size)
print("ZIP_SHA256 =", sha256_file(ZIP_PATH))

print("\n14.15D1_WHEELHOUSE_BUILT")
print("OFFLINE_INSTALL_VERIFIED=NO")
print("GT_READ=NO")
print("INFERENCE_RUN=NO")
print("TRACKING_RUN=NO")
REBUILT_ZIP_SHA = sha256_file(ZIP_PATH)
print("REBUILT_ZIP_SHA_FOR_PERSISTENCE =", REBUILT_ZIP_SHA)

REBUILT_MANIFEST_SHA = sha256_file(manifest_path)


In [ ]:
# New read-only Kaggle Cell. Not executed against a Kaggle wheelhouse.
# Set ZIP_PATH to a recovered/rebuilt ZIP; do not run when it is absent.
from pathlib import Path, PurePosixPath
from email.parser import BytesParser
import hashlib
import io
import json
import zipfile

ZIP_PATH = Path('/kaggle/working/biohub_wheelhouse_py312_v1.zip')
# Original artifact locks. A rebuild is a new artifact unless these match.
EXPECTED_ZIP_SHA = REBUILT_ZIP_SHA  # New ZIP identity, not the historical ZIP lock.
print('NEW_ZIP_IDENTITY', EXPECTED_ZIP_SHA)
EXPECTED_MANIFEST_SHA = REBUILT_MANIFEST_SHA  # New manifest; original evidence remains unchanged.

def stream_sha(stream):
    h = hashlib.sha256()
    for block in iter(lambda: stream.read(1024 * 1024), b''):
        h.update(block)
    return h.hexdigest()

assert ZIP_PATH.is_file(), 'ARTIFACT_MISSING: recover/rebuild once; do not repeat this check'
with ZIP_PATH.open('rb') as f:
    actual_zip_sha = stream_sha(f)
assert actual_zip_sha == EXPECTED_ZIP_SHA, ('NEW_OR_CHANGED_ZIP', actual_zip_sha)
with zipfile.ZipFile(ZIP_PATH) as archive:
    infos = archive.infolist()
    assert len(infos) == len({i.filename for i in infos}), 'Duplicate ZIP entries'
    for i in infos:
        p = PurePosixPath(i.filename)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in i.filename
        assert ((i.external_attr >> 16) & 0o170000) != 0o120000, 'ZIP symlink'
    files = {i.filename for i in infos if not i.is_dir()}
    print('DIRECTORY_ENTRIES', [i.filename for i in infos if i.is_dir()])
    raw_manifest = archive.read('wheelhouse_manifest.json')
    assert hashlib.sha256(raw_manifest).hexdigest() == EXPECTED_MANIFEST_SHA
    manifest = json.loads(raw_manifest)
    records = manifest['wheels']
    assert len(records) == 13 and len({r['filename'] for r in records}) == 13
    assert files == {'wheelhouse_manifest.json'} | {'wheels/' + r['filename'] for r in records}
    for r in records:
        assert PurePosixPath(r['filename']).name == r['filename']
        member = 'wheels/' + r['filename']
        assert archive.getinfo(member).file_size == r['size_bytes']
        with archive.open(member) as f:
            assert stream_sha(f) == r['sha256'], member
        with zipfile.ZipFile(io.BytesIO(archive.read(member))) as wheel:
            names = [n for n in wheel.namelist() if n.endswith('.dist-info/METADATA')]
            assert len(names) == 1
            metadata = BytesParser().parsebytes(wheel.read(names[0]))
            print('WHEEL_PASS', r['filename'], r['size_bytes'], r['sha256'])
            print('METADATA', metadata['Name'], metadata['Version'], metadata['Requires-Python'])
            for requirement in metadata.get_all('Requires-Dist', []):
                print('REQUIRES', requirement)
    assert archive.testzip() is None
print('WHEELHOUSE_ZIP_INTEGRITY_PASS')
print('OFFLINE_INSTALL_VERIFIED=NO; IMPORT_SMOKE_VERIFIED=NO; FILES_WRITTEN=0')

print('VERIFIED_WHEELHOUSE_LOCK — persist and paste this block into notebook 02:')
print('WHEELHOUSE_LOCK = ' + repr({'zip_sha256': actual_zip_sha, 'manifest_sha256': EXPECTED_MANIFEST_SHA}))
print('NEW_ARTIFACT_IDENTITY; OFFLINE_RESTORE_STILL_NOT_VERIFIED')
